# Capstone FDE Portfolio Project — Applied

**FDE Delivery · Week 24b**

Offline notebook for the final capstone: Enterprise AI SDLC Assistant architecture, DiscoveryBrief, prompt contracts, offline pipeline, eval regression, go-live checklist, README, and interview pitch.

## 1. C4 architecture

```mermaid
C4Container
  title Enterprise AI SDLC Assistant
  Person(po, "Product Owner", "Reviews generated backlog")
  Container(ui, "Human Review UI", "Streamlit/Next.js", "Review cited JSON")
  Container(api, "FastAPI Backend", "Pydantic v2", "OpenAPI, auth, validation")
  Container(rag, "RAG Pipeline", "Python", "retrieve, rerank, prompt, validate, cite")
  ContainerDb(pg, "pgvector", "Postgres", "project knowledge")
  Container(eval, "Eval Harness", "CI", "golden-set regression")
  po --> ui
  ui --> api
  api --> rag
  rag --> pg
  eval --> api
```

The capstone wraps the model path with human review, prompt registry, eval, guardrails, observability, and deployment evidence.

## 2. DiscoveryBrief and MVP scope

In [ ]:
from __future__ import annotations
import json
from statistics import mean
from typing import Literal, Protocol
from pydantic import BaseModel, ConfigDict, Field, ValidationError
print('Week 24b capstone notebook ready')
discovery = {
 'BLUF': 'Cut requirement-to-backlog time from 3-5 days to 4 hours at ~$0.15/PBI with groundedness >=90% on 250 golden items.',
 'JTBD': 'When a new requirements document arrives, create a cited draft backlog and test suite so leads review decisions rather than transcribe work.',
 'stakeholders': ['Product Owner','Engineering Manager','QA Lead','Platform Engineering','Compliance'],
 'SMART': ['3-5 days -> 4 hours within two release trains','groundedness >=90%','100% schema-valid JSON','cost <=$0.20/PBI target $0.15','p95 generation <=120s','citations on 100% generated items'],
 'constraints': ['Jira Cloud integration target','no source code shipped to third-party providers','OAuth/OIDC + RBAC','PII redaction','human review before export']}
for k,v in discovery.items(): print(k, ':', v)
scope = {'Must':['upload requirement doc','ingest project knowledge','generate epic/features/PBIs/tests','RAG citations','JSON schema validation','human review UI','source refs','cost+latency logs'], 'Should':['evaluation dashboard','API access','prompt registry','OpenAPI docs'], 'Could':['batch uploads','semantic cache'], 'Wont':['automatic Jira submission','autonomous backlog acceptance']}
print('\nMoSCoW scope')
for k,v in scope.items(): print(k, '->', ', '.join(v))

## 3. Prompt contracts as first-class data

In [ ]:
prompt_contracts = [
 {'id':'generate_epic.v1','purpose':'Generate one cited Epic from requirements and retrieved project context','model':'gpt-4o','input_schema':['requirement_text','project_context_chunks','product_area','constraints','definition_of_ready'],'output_schema':'Epic{title,description,acceptance_criteria[],estimated_effort,dependencies[],source_citations[],clarification_needed,clarification_questions[]}','safety':['cite every claim','ask clarification instead of hallucinating','JSON only'],'changelog':['v1 initial schema and citation contract']},
 {'id':'generate_functional_test.v2','purpose':'Generate cited functional tests for a PBI','model':'gpt-4o','input_schema':['pbi','acceptance_criteria','retrieved_chunks','test_conventions'],'output_schema':'TestCase{name,preconditions,steps[],expected_results[],category,source_citations[],automation_candidate}','safety':['positive, negative, and edge paths','ground every expected result'],'changelog':['v2 improved negative-path coverage; groundedness +2pp; cost -8%']}
]
for pc in prompt_contracts:
    print('\n#', pc['id'])
    for k,v in pc.items():
        if k != 'id': print('-', k, ':', v)

## 4. Offline SDLC pipeline

In [ ]:
class Citation(BaseModel): chunk_id: str; quote: str
class TestCase(BaseModel):
    model_config = ConfigDict(extra='forbid')
    title: str; category: Literal['functional','non_functional','automation']; steps: list[str]; expected_result: str; source_citations: list[Citation] = Field(min_length=1)
    metric: str | None = None; threshold: str | None = None; measurement_method: str | None = None
class PBI(BaseModel):
    model_config = ConfigDict(extra='forbid')
    title: str; kind: Literal['functional','non_functional']; acceptance_criteria: list[str]; source_citations: list[Citation] = Field(min_length=1); test_cases: list[TestCase]
class Feature(BaseModel):
    model_config = ConfigDict(extra='forbid')
    title: str; description: str; source_citations: list[Citation] = Field(min_length=1); pbis: list[PBI]
class Epic(BaseModel):
    model_config = ConfigDict(extra='forbid')
    title: str; description: str; acceptance_criteria: list[str]; estimated_effort: Literal['S','M','L']; dependencies: list[str] = Field(default_factory=list); source_citations: list[Citation] = Field(min_length=1); clarification_needed: bool = False; clarification_questions: list[str] = Field(default_factory=list); features: list[Feature] = Field(default_factory=list)
class Chunk(BaseModel): chunk_id: str; text: str; metadata: dict[str,str]
class RAGCorpus:
    def __init__(self, chunks): self.chunks=chunks
    def retrieve(self, q, k=5):
        terms={t.lower().strip('.,:;') for t in q.split() if len(t)>3}; scored=[]
        for c in self.chunks:
            s=sum(t in c.text.lower() for t in terms)
            if s: scored.append((s,c))
        return [c for _,c in sorted(scored,key=lambda x:x[0], reverse=True)[:k]] or self.chunks[:k]
class FakeLLMProvider:
    def generate_json(self, prompt, chunks):
        cite={'chunk_id':chunks[0].chunk_id,'quote':chunks[0].text[:90]}
        if 'TBD' in prompt or 'some reports' in prompt.lower():
            return {'title':'Clarification needed before backlog generation','description':'Requirement lacks actor/workflow/success detail.','acceptance_criteria':['Clarify user and outcome.'],'estimated_effort':'S','source_citations':[cite],'clarification_needed':True,'clarification_questions':['Who is the user?','What is success?'],'features':[]}
        tc={'title':'Approval path stores audit','category':'functional','steps':['Generate','Approve','Inspect audit'],'expected_result':'Approved item has reviewer and citations.','source_citations':[cite]}
        nft={'title':'Latency target measured','category':'non_functional','steps':['Run 20 generations'],'expected_result':'p95 within target','metric':'p95_latency_seconds','threshold':'<=120','measurement_method':'OTel','source_citations':[cite]}
        auto={'title':'Schema validation automation','category':'automation','steps':['POST /generate','Validate schema'],'expected_result':'JSON schema valid','source_citations':[cite]}
        pbi={'title':'Reviewer approves generated PBI','kind':'functional','acceptance_criteria':['All generated items have citations.','Approval precedes export.'],'source_citations':[cite],'test_cases':[tc,nft,auto]}
        return {'title':'AI-assisted requirements-to-backlog workflow','description':'Convert requirements into cited reviewable backlog JSON.','acceptance_criteria':['Schema-valid output','Citations everywhere','Human approval before export'],'estimated_effort':'M','dependencies':['Jira/Azure DevOps field mapping'],'source_citations':[cite],'features':[{'title':'Human-reviewed backlog generation','description':'Review cited PBIs before export.','source_citations':[cite],'pbis':[pbi]}]}
def corpus(): return RAGCorpus([Chunk(chunk_id='proj-001',text='All backlog items require citations and human approval before export.',metadata={}), Chunk(chunk_id='proj-002',text='Non-functional tests must name metric threshold and measurement method.',metadata={}), Chunk(chunk_id='proj-003',text='Cost cap is 0.20 USD per generated PBI.',metadata={})])
def sdlc_pipeline(text):
    c=corpus(); chunks=c.retrieve(text); raw=FakeLLMProvider().generate_json(text, chunks); return Epic.model_validate(raw)
for label, text in [('clean','Product Owner uploads requirements and reviews generated PBIs with citations.'),('underspecified','TBD add AI for some reports later')]:
    print('\n###', label); print(sdlc_pipeline(text).model_dump_json(indent=2))
try: Epic.model_validate({'title':'bad','description':'no citations','acceptance_criteria':[],'estimated_effort':'XL'})
except ValidationError as e: print('schema failure caught:', e.errors()[0]['msg'])

## 5. Evaluation harness and regression report

In [ ]:
class VersionSummary(BaseModel):
    version: str; groundedness: float; citation_coverage: float; pbi_f1: float; test_coverage: float; cost_per_pbi: float; latency_ms: int; verdict: Literal['PROMOTE','HOLD','ROLLBACK']; notes: str
class RegressionReport(BaseModel):
    prompt_id: str; versions: list[VersionSummary]
    def render(self):
        vals=[v.groundedness for v in self.versions]; blocks='▁▂▃▄▅▆▇█'; lo=min(vals); hi=max(vals); spark=''.join(blocks[int((v-lo)/(hi-lo or 1)*(len(blocks)-1))] for v in vals)
        lines=[f'# Regression Report — {self.prompt_id}', f'Groundedness trend: {spark}', '| Version | Grounded | Citation | PBI F1 | Test | Cost/PBI | Latency | Verdict | Notes |','|---|---:|---:|---:|---:|---:|---:|---|---|']
        for v in self.versions: lines.append(f'| {v.version} | {v.groundedness:.1f}% | {v.citation_coverage:.1f}% | {v.pbi_f1:.2f} | {v.test_coverage:.1f}% | ${v.cost_per_pbi:.3f} | {v.latency_ms} | **{v.verdict}** | {v.notes} |')
        return '\n'.join(lines)
versions=[VersionSummary(version='v1',groundedness=78,citation_coverage=89,pbi_f1=.72,test_coverage=74,cost_per_pbi=.150,latency_ms=92000,verdict='HOLD',notes='baseline below quality target'), VersionSummary(version='v2',groundedness=84,citation_coverage=91,pbi_f1=.78,test_coverage=80,cost_per_pbi=.173,latency_ms=101000,verdict='HOLD',notes='+15% cost requires review'), VersionSummary(version='v3',groundedness=86,citation_coverage=95,pbi_f1=.83,test_coverage=85,cost_per_pbi=.150,latency_ms=89000,verdict='PROMOTE',notes='quality lift at baseline cost')]
print(RegressionReport(prompt_id='generate_functional_test', versions=versions).render())

## 6. Go-live checklist — CONDITIONAL_GO

In [ ]:
green = ['auth configured','RBAC roles mapped','Pydantic schemas stable','OpenAPI generated','PII redaction enabled','input classifier enabled','output validator enabled','prompt registry versioned','250 golden set seeded','schema validity 100%','citations required','audit log writes','OTel cost spans','latency dashboard','docker compose works','unit tests pass','API smoke test pass','human review queue works','README has BLUF','demo script recorded']
amber = ['evaluation dashboard needs v1.1 trend slices','cloud deploy needs one more dry-run','cost dashboard needs per-tenant attribution']
red = ['missing DPA template for enterprise customers','missing rollback drill evidence']
print('Go-live verdict: CONDITIONAL_GO')
print('green', len(green), green)
print('amber', len(amber), amber)
print('red', len(red), red)
print('next steps: close DPA template, run rollback drill, then repeat checklist before 100% traffic')

## 7. README opening and 90-second pitch

## Exercises
1. Add one golden-set item for an API-rate-limit requirement and score expected PBIs.
2. Write an ADR for Streamlit vs Next.js.
3. Add a schema migration test that would catch a renamed Jira field.
4. Rewrite the capstone pitch for a customer CTO instead of a hiring manager.

## Links
- Literature note: `02 Literature Notes/FDE Delivery/Capstone FDE Portfolio Project — Applied`
- Snippets: `04 Code Snippets/FDE Delivery/FDE Week 24b Enterprise AI SDLC Assistant Offline Pipeline`, `.../FDE Week 24b Capstone Evaluation Harness Regression Report`
- MOC: `06 Maps of Content/FDE Delivery Concepts`

In [ ]:
readme = """# Enterprise AI SDLC Assistant

**BLUF:** Converts requirement docs into cited, schema-valid epics, features, PBIs, and test cases, cutting requirement-to-reviewed-backlog time from 3-5 days to 4 hours at ~$0.15/PBI and 92% groundedness on a 250-item golden set.

![Groundedness across prompt versions](docs/assets/groundedness-chart-placeholder.png)

This FDE capstone ingests requirements and project knowledge, uses RAG and prompt contracts to generate structured backlog JSON, and keeps humans in review before Jira/Azure DevOps import.

[3-minute demo video](docs/demo-video-placeholder.md)

| Metric | Result |
|---|---:|
| Groundedness | 92% |
| Hallucination rate | 1.4% |
| Avg cost/PBI | $0.15 |
| Avg latency / epic run | 90s |

Run it in 15 minutes: `make setup`, `make test`, `make eval`, `make run`.
"""
pitch = """I built an Enterprise AI SDLC Assistant that turns PRDs into Jira-ready backlog and tests. The headline result is 3-5 days down to 4 hours with 92% groundedness on a 250-item golden set. The hardest problem was preventing plausible but ungrounded PBIs; I solved it with strict schemas, citations, human review, and eval gates. The key tradeoff was keeping Jira submission human-approved in v1 to preserve trust."""
print(readme)
print('90-second pitch:', pitch)